# 데이터분석 순서
* 데이터세트선택 : CSV, EXCEL, DB에서 데이터를 읽어옴
* 데이터 전처리 : 데이터타입, 결측값, 이상치탐지, 데이터분포분석, 상관관계
* 데이터 변환(특성추출) : 원본 데이터에서 새로운 데이터 생성, 삭제, 스케일링, 구간화
* 데이터 마이닝(모델만들기, 분석) : 분석에 적합한 알고리즘 선택, 모델 생성, 튜닝
* 결과 평가 : 테스트 데이터를 이용해서 데이터 마이닝으로 만든 모델의 성능 평가
* KDD 분석 방법론

# 데이터 전처리
* 데이터 타입 변환
* 결측치 탐지 및 보관
* 이상치 탐지 및 처리
* 데이터 특성 파악(치우침, 분포 특성)
* 변수들 간의 상관관계 분석

In [1]:
import pandas as pd
import numpy as np

# 1. 데이터 세트 선택 및 로딩
* 데이터 로드 후 head(), tail()로 컬럼과 데이터 파악


In [4]:
data = pd.read_csv("./data/Taitanic_train.csv")

# 2. .info()로 컬럼명, 결측치, 데이터 타입 파악

In [4]:
# 데이터가 너무 커서 Non-Null 표시가 안될 때 show_counts=True 로 파악
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


# 3. describe()로 기초통계량 파악(이상치 파악 위해)
* 극단값 파악

# 4. 결측값 보고 비율보고 대치 및 삭제하기
* 결측값 비율 계산 : isna().sum() / len(데이터프레임) * 100
* 결측값 비율이 5% 미만인 경우 : 행, 열 제거. 분석에 크게 영향을 미치지 않음.
* 결측값 비율이 5% ~ 30% : 결측값을 대체(Imputation)
    * 수치형 데이터(숫자형, 나이, 가격) : 평균(mean), 중앙값(median), 최빈값(mode)
    * 범주형 데이터(문자형or(숫자형), 선실등급, 탑승지 : 최빈값(mode)으로 대체
* 결측값 비율이 30% ~ 50% : 컬럼의 중요도에 따라 결측값을 대체 혹은 삭제
    * KNN(K-Nearlist Neighbor, 최근접이웃) imputer, 회귀분석을 통해 결측값 대체
* 결측값 비율이 50% 이상 : 해당 컬럼 삭제

In [7]:
data.isna().sum() / len(data) * 100

PassengerId     0.000000
Survived        0.000000
Pclass          0.000000
Name            0.000000
Sex             0.000000
Age            19.865320
SibSp           0.000000
Parch           0.000000
Ticket          0.000000
Fare            0.000000
Cabin          77.104377
Embarked        0.224467
dtype: float64

### 결측치 처리방법
* 1) 단순대치법(simple imputation)
    * (1) 완전분석 : 결측값이 있는 모든 행을 삭제하고 완전한 자료만으로 분석(데이터 손실이 너무 커서 잘 쓰지 않음)
        * 결측값을 삭제해도 모델을 만들기에 충분히 많은 데이터가 있는 경우
        * 결측값을 삭제한 후에 데이터에 편향이 없다는 전제가 있을 경우
        * dropna() : 결측이 있는 모든 행 삭제

In [8]:
data.dropna()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,51.8625,E46,S
10,11,1,3,"Sandstrom, Miss. Marguerite Rut",female,4.0,1,1,PP 9549,16.7000,G6,S
11,12,1,1,"Bonnell, Miss. Elizabeth",female,58.0,0,0,113783,26.5500,C103,S
...,...,...,...,...,...,...,...,...,...,...,...,...
871,872,1,1,"Beckwith, Mrs. Richard Leonard (Sallie Monypeny)",female,47.0,1,1,11751,52.5542,D35,S
872,873,0,1,"Carlsson, Mr. Frans Olof",male,33.0,0,0,695,5.0000,B51 B53 B55,S
879,880,1,1,"Potter, Mrs. Thomas Jr (Lily Alexenia Wilson)",female,56.0,0,1,11767,83.1583,C50,C
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S


* (2)  평균 대치법 : 결측치가 있는 컬럼에서 데이터의 평균을 구한 후 결측값을 대치
    * 평균을 이용하기 때문에 간편
    * 데이터에 이상치가 있을 경우 평균을 이용할 수 없다
    * 데이터에 이상치가 있는 경우 중앙값이나 최빈값을 고려해야 한다

In [16]:
a = pd.Series([24, 5, 10, 34, 20, 18, 28, 20])
b = pd.Series([24, 5, 10, 34, 20, 18, 28, 2000])

In [17]:
a.mean()

np.float64(19.875)

In [18]:
b.mean()

np.float64(267.375)

In [19]:
a.median()

np.float64(20.0)

In [20]:
sorted(b)

[5, 10, 18, 20, 24, 28, 34, 2000]

Age 컬럼의 결측값을 평균 대치법으로 대치

In [6]:
data['Age'].isna().sum()

np.int64(177)

In [7]:
data['Age'].describe()

count    714.000000
mean      29.699118
std       14.526497
min        0.420000
25%       20.125000
50%       28.000000
75%       38.000000
max       80.000000
Name: Age, dtype: float64

In [8]:
data['Age'].fillna(data['Age']).mean()

np.float64(29.69911764705882)

In [ ]:
age_na_idx = data['Age'].fillna()

In [9]:
# 깊은 복사, 얕은 복사
data2 = data.copy()
data3 = data.copy()

In [ ]:
data['Age'] = data['Age'].fillna(data['Age'].mean())

In [ ]:
data.loc[age_na_idx]

# scikit-learn의 simple imputer를 이용한 대치

In [10]:
from sklearn.impute import SimpleImputer

In [11]:
imp_mean = SimpleImputer(strategy='mean')
imp_mean.fit_transform(data['Age'].values.reshape(-1,1))

array([[22.        ],
       [38.        ],
       [26.        ],
       [35.        ],
       [35.        ],
       [29.69911765],
       [54.        ],
       [ 2.        ],
       [27.        ],
       [14.        ],
       [ 4.        ],
       [58.        ],
       [20.        ],
       [39.        ],
       [14.        ],
       [55.        ],
       [ 2.        ],
       [29.69911765],
       [31.        ],
       [29.69911765],
       [35.        ],
       [34.        ],
       [15.        ],
       [28.        ],
       [ 8.        ],
       [38.        ],
       [29.69911765],
       [19.        ],
       [29.69911765],
       [29.69911765],
       [40.        ],
       [29.69911765],
       [29.69911765],
       [66.        ],
       [28.        ],
       [42.        ],
       [29.69911765],
       [21.        ],
       [18.        ],
       [14.        ],
       [40.        ],
       [27.        ],
       [29.69911765],
       [ 3.        ],
       [19.        ],
       [29

# 실제 메모상의 주소를 출력 id()

In [ ]:
print("data의 메모리 주소: ", id(data))
print("data_reassigned의 메모리 주소: ", id(data_reassigned))
print("data_copyed의 메모리 주소: ", id(data_copyed))